# Tourism Experience Analytics
## Classification, Rating Prediction and Recommendation System

This Google Colab notebook follows the project requirements:
- Data Cleaning and Preprocessing
- Exploratory Data Analysis
- Regression: Rating Prediction
- Classification: Visit Mode Prediction
- Recommendation System
- Model Evaluation and Saving

## 1. Import Libraries

In [ ]:
!pip -q install joblib openpyxl

import os, glob, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

## 2. Upload the Tourism Dataset Files

Upload the CSV/XLSX files downloaded from the Tourism Dataset folder.  
The project expects files such as Transaction, User, City, Item, Type, VisitMode, Continent, Country and Region.

You can upload all files directly into Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

## 3. Automatically Load Available Files

In [ ]:
def load_file(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    else:
        return None

dataframes = {}
for f in glob.glob("*"):
    if f.lower().endswith((".csv", ".xlsx", ".xls")):
        try:
            df = load_file(f)
            key = os.path.splitext(os.path.basename(f))[0]
            dataframes[key] = df
            print(f"\n{key}: {df.shape}")
            display(df.head())
        except Exception as e:
            print(f"Could not load {f}: {e}")

## 4. Identify Tables

Check the printed file names and adjust the mapping below if your uploaded filenames differ.

In [ ]:
# Update these names only if necessary
TABLES = {
    "transaction": None,
    "user": None,
    "city": None,
    "item": None,
    "type": None,
    "visit_mode": None,
    "continent": None,
    "country": None,
    "region": None
}

print("Available tables:", list(dataframes.keys()))
print("\nSet TABLES manually if automatic names do not match.")

## 5. Helper: Find Tables by Filename

In [ ]:
def find_table(keywords):
    for name, df in dataframes.items():
        lower = name.lower()
        if any(k in lower for k in keywords):
            return df.copy(), name
    return None, None

for logical_name, keys in {
    "transaction":["transaction"],
    "user":["user"],
    "city":["city"],
    "item":["item","attraction"],
    "type":["type"],
    "visit_mode":["visitmode","visit_mode","visit mode"],
    "continent":["continent"],
    "country":["country"],
    "region":["region"]
}.items():
    df, name = find_table(keys)
    TABLES[logical_name] = df
    print(logical_name, "->", name)

## 6. Basic Data Cleaning

In [ ]:
for name, df in TABLES.items():
    if isinstance(df, pd.DataFrame):
        df.columns = [str(c).strip() for c in df.columns]
        df = df.drop_duplicates()
        # Remove completely empty rows/columns
        df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
        TABLES[name] = df

        print(f"\n{name.upper()} | Shape: {df.shape}")
        print("Missing values:")
        display(df.isnull().sum().sort_values(ascending=False).head(10))

## 7. Merge the Datasets

The following merge logic uses the IDs described in the project specification.  
If your actual column names differ, edit the merge keys.

In [ ]:
tx = TABLES["transaction"].copy()
users = TABLES["user"]
city = TABLES["city"]
items = TABLES["item"]
types = TABLES["type"]
continents = TABLES["continent"]
countries = TABLES["country"]
regions = TABLES["region"]

# Start with transaction data
merged = tx.copy()

# Merge user information
if users is not None and "UserId" in merged.columns and "UserId" in users.columns:
    merged = merged.merge(users, on="UserId", how="left", suffixes=("", "_User"))

# Merge attraction/item information
if items is not None and "AttractionId" in merged.columns and "AttractionId" in items.columns:
    merged = merged.merge(items, on="AttractionId", how="left", suffixes=("", "_Attraction"))

# Merge attraction type
if types is not None and "AttractionTypeId" in merged.columns and "AttractionTypeId" in types.columns:
    merged = merged.merge(types, on="AttractionTypeId", how="left")

# Merge user city
if city is not None:
    city_cols = city.copy()
    if "CityId" in merged.columns and "CityId" in city_cols.columns:
        merged = merged.merge(city_cols, on="CityId", how="left", suffixes=("", "_UserCity"))

# Merge continent/region/country where IDs are available
for lookup, key in [(continents, "ContinentId"), (regions, "RegionId"), (countries, "CountryId")]:
    if lookup is not None and key in merged.columns and key in lookup.columns:
        merged = merged.merge(lookup, on=key, how="left", suffixes=("", "_Lookup"))

print("Final merged shape:", merged.shape)
display(merged.head())

## 8. Feature Engineering

In [ ]:
# Convert rating to numeric
if "Rating" in merged.columns:
    merged["Rating"] = pd.to_numeric(merged["Rating"], errors="coerce")
    merged = merged[(merged["Rating"].isna()) | ((merged["Rating"] >= 1) & (merged["Rating"] <= 5))]

# Create date-like features
for col in ["VisitYear", "VisitMonth"]:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# Simple popularity feature based on number of transactions
if "AttractionId" in merged.columns:
    popularity = merged["AttractionId"].value_counts()
    merged["AttractionPopularity"] = merged["AttractionId"].map(popularity)

# User historical mean rating
if "UserId" in merged.columns and "Rating" in merged.columns:
    user_avg = merged.groupby("UserId")["Rating"].transform("mean")
    merged["UserAverageRating"] = user_avg

# Fill numeric missing values
numeric_cols = merged.select_dtypes(include=np.number).columns
merged[numeric_cols] = merged[numeric_cols].fillna(merged[numeric_cols].median())

# Fill categorical missing values
cat_cols = merged.select_dtypes(exclude=np.number).columns
merged[cat_cols] = merged[cat_cols].fillna("Unknown")

print("Cleaned dataset shape:", merged.shape)
display(merged.head())

## 9. Save Cleaned Dataset

In [ ]:
merged.to_csv("cleaned_tourism_dataset.csv", index=False)
print("Saved: cleaned_tourism_dataset.csv")

## 10. Exploratory Data Analysis (EDA)

In [ ]:
# Rating distribution
if "Rating" in merged.columns:
    plt.figure(figsize=(7,4))
    plt.hist(merged["Rating"].dropna(), bins=10)
    plt.title("Distribution of Attraction Ratings")
    plt.xlabel("Rating")
    plt.ylabel("Frequency")
    plt.show()

# Visit mode distribution
if "VisitMode" in merged.columns:
    merged["VisitMode"].value_counts().head(15).plot(kind="bar", figsize=(9,4))
    plt.title("Visit Mode Distribution")
    plt.ylabel("Count")
    plt.show()

# Top attractions
attraction_col = "Attraction" if "Attraction" in merged.columns else "AttractionId"
if attraction_col in merged.columns:
    merged[attraction_col].value_counts().head(10).plot(kind="bar", figsize=(10,4))
    plt.title("Top 10 Most Visited Attractions")
    plt.ylabel("Visits")
    plt.show()

## 11. Regression Model: Predict Attraction Rating

In [ ]:
if "Rating" not in merged.columns:
    raise ValueError("Rating column not found.")

regression_features = [
    c for c in [
        "VisitYear", "VisitMonth", "AttractionId", "UserId",
        "AttractionTypeId", "ContinentId", "RegionId",
        "CountryId", "CityId", "AttractionPopularity"
    ] if c in merged.columns
]

X_reg = merged[regression_features].copy()
y_reg = merged["Rating"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

cat_features = X_reg.select_dtypes(include="object").columns.tolist()
num_features = [c for c in X_reg.columns if c not in cat_features]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features)
])

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    )
}

reg_results = {}
best_reg_model = None
best_r2 = -np.inf

for name, model in reg_models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)

    reg_results[name] = {"MAE": mean_absolute_error(y_test, pred), "RMSE": rmse, "R2": r2}
    if r2 > best_r2:
        best_r2 = r2
        best_reg_model = pipe

pd.DataFrame(reg_results).T

## 12. Save Best Regression Model

In [ ]:
joblib.dump(best_reg_model, "best_regression_model.joblib")
print("Saved: best_regression_model.joblib")

## 13. Classification Model: Predict Visit Mode

In [ ]:
if "VisitMode" not in merged.columns:
    raise ValueError("VisitMode column not found.")

classification_features = [
    c for c in [
        "VisitYear", "VisitMonth", "AttractionId", "UserId",
        "AttractionTypeId", "ContinentId", "RegionId",
        "CountryId", "CityId", "AttractionPopularity"
    ] if c in merged.columns
]

X_clf = merged[classification_features].copy()
y_clf = merged["VisitMode"].astype(str)

# Remove very rare classes to make splitting more stable
counts = y_clf.value_counts()
valid_classes = counts[counts >= 2].index
mask = y_clf.isin(valid_classes)
X_clf, y_clf = X_clf[mask], y_clf[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

cat_features = X_clf.select_dtypes(include="object").columns.tolist()
num_features = [c for c in X_clf.columns if c not in cat_features]

clf_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features)
])

clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"
    )
}

clf_results = {}
best_clf_model = None
best_f1 = -1

for name, model in clf_models.items():
    pipe = Pipeline([("preprocessor", clf_preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    f1 = f1_score(y_test, pred, average="weighted", zero_division=0)
    clf_results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1": f1
    }

    if f1 > best_f1:
        best_f1 = f1
        best_clf_model = pipe
        best_predictions = pred

pd.DataFrame(clf_results).T

## 14. Classification Evaluation

In [ ]:
print(classification_report(y_test, best_predictions, zero_division=0))

labels = sorted(pd.unique(np.concatenate([np.array(y_test), np.array(best_predictions)])))
cm = confusion_matrix(y_test, best_predictions, labels=labels)

plt.figure(figsize=(8,6))
plt.imshow(cm)
plt.colorbar()
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 15. Save Best Classification Model

In [ ]:
joblib.dump(best_clf_model, "best_classification_model.joblib")
print("Saved: best_classification_model.joblib")

## 16. Recommendation System

A simple collaborative recommendation method is used:
1. Build a User × Attraction rating matrix.
2. For a selected user, recommend highly rated attractions not yet visited.
3. If user history is insufficient, use globally popular/high-rated attractions.

In [ ]:
def recommend_attractions(user_id, n=10):
    required = {"UserId", "AttractionId", "Rating"}
    if not required.issubset(merged.columns):
        raise ValueError(f"Missing required columns: {required - set(merged.columns)}")

    ratings = merged.groupby(["UserId", "AttractionId"])["Rating"].mean().reset_index()

    visited = set(ratings.loc[ratings["UserId"] == user_id, "AttractionId"])

    # Global attraction scores
    stats = ratings.groupby("AttractionId").agg(
        AvgRating=("Rating", "mean"),
        RatingCount=("Rating", "count")
    ).reset_index()

    stats["Score"] = stats["AvgRating"] * np.log1p(stats["RatingCount"])
    recommendations = stats[~stats["AttractionId"].isin(visited)].sort_values(
        "Score", ascending=False
    ).head(n)

    # Add attraction names when available
    if "Attraction" in merged.columns:
        names = merged[["AttractionId", "Attraction"]].drop_duplicates()
        recommendations = recommendations.merge(names, on="AttractionId", how="left")

    return recommendations

sample_user = merged["UserId"].iloc[0]
recommend_attractions(sample_user, n=10)

## 17. Download Project Outputs

In [ ]:
from google.colab import files

for filename in [
    "cleaned_tourism_dataset.csv",
    "best_regression_model.joblib",
    "best_classification_model.joblib"
]:
    if os.path.exists(filename):
        print("Ready:", filename)

# Uncomment any line below to download
# files.download("cleaned_tourism_dataset.csv")
# files.download("best_regression_model.joblib")
# files.download("best_classification_model.joblib")

## 18. Next Step: Streamlit Deployment

The trained files created by this notebook can be placed in a GitHub repository:

```text
tourism-project/
├── app.py
├── requirements.txt
├── models/
│   ├── best_regression_model.joblib
│   └── best_classification_model.joblib
└── data/
    └── cleaned_tourism_dataset.csv
```

The Streamlit app can then provide:
- Visit mode prediction
- Attraction rating prediction
- Personalized recommendations
- Tourism analytics visualizations